# 12. Presentation Evidence Retrieval

## Goal

Load real manual-test PPTX and PDF presentations, preserve slide/page citations, and search them together with transcript evidence. Canva presentations use the same PDF or PPTX export path.

## Why This Step Matters

Meetings often reference a slide deck. Combining presentation evidence with the transcript lets an answer point to both what was presented and what was actually said.

## Inputs

- `outputs/manual_test/presentations/meeting_budget_sample.pptx`
- `outputs/manual_test/presentations/meeting_budget_sample.pdf`
- one transcript-only evidence row

## Outputs

Evidence rows containing `source`, `filename`, `location`, and `text`, followed by Thai query results with slide, page, or transcript citations.

## Setup

In [13]:
import shutil
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.documents.presentation import load_presentation
from src.retrieval.chunking import TranscriptChunk
from src.retrieval.search import search_chunks

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF reader: {shutil.which('pdftotext')}")

Project root: /home/lataeq/Code/Saranote
PDF reader: /usr/bin/pdftotext


## Load Data

Load the visible manual-test files created from the same Thai budget meeting content as the audio sample. The generated binary files stay under ignored `outputs/`.

In [14]:
PRESENTATION_DIR = PROJECT_ROOT / "outputs" / "manual_test" / "presentations"
pptx_path = PRESENTATION_DIR / "meeting_budget_sample.pptx"
pdf_path = PRESENTATION_DIR / "meeting_budget_sample.pdf"
assert pptx_path.exists() and pdf_path.exists(), "Manual-test presentation files are missing"

pptx_rows = load_presentation(pptx_path)
pdf_rows = load_presentation(pdf_path)
presentation_rows = pptx_rows + pdf_rows
for label, path, rows in (("PPTX", pptx_path, pptx_rows), ("PDF", pdf_path, pdf_rows)):
    unit = "slides" if label == "PPTX" else "pages"
    print(f"{label}: {path.name} ({len(rows)} {unit})")
    for row in rows:
        print(f"  {row['location']}: {row['text'][:70]}")

PPTX: meeting_budget_sample.pptx (3 slides)
  slide-01: THAIMEETSCRIBE / TEST DECK ประชุมงบประมาณ แผนงบประมาณ โครงการใหม่ ข้อส
  slide-02: มติที่ประชุม ที่ประชุมปรับงบเพื่อให้งานอบรมเดินหน้า 01 ลดค่าใช้จ่าย ด้
  slide-03: งานที่มอบหมาย สองงานต้องเสร็จก่อนส่งรายงาน 01 ฝ่ายดำเนินงาน ส่งเอกสารง
PDF: meeting_budget_sample.pdf (3 pages)
  page-01: THAIMEETSCRIBE / TEST DECK ประชุมงบประมาณ 01 ปรับแผนงบประมาณ แผนงบประม
  page-02: มติที่ประชุม ที่ประชุมปรับงบเพื่อให้งานอบรมเดินหน้า 01 02 03 ลดค่าใช้จ
  page-03: งานที่มอบหมาย สองงานต้องเสร็จก่อนส่งรายงาน 01 02 ฝ่ายดำเนินงาน ทุกฝ่าย


## Combine Evidence

Add a transcript-only statement that does not appear in the presentation, then keep all sources in one evidence collection.

In [15]:
transcript_rows = [
    {
        "source": "transcript",
        "filename": "meeting_test.wav",
        "location": "transcript-05",
        "text": "ที่ประชุมยังพูดถึงความเสี่ยงเรื่องข้อมูลไม่ครบ",
    }
]
evidence_rows = presentation_rows + transcript_rows
print(f"Combined evidence rows: {len(evidence_rows)}")
print(pd.Series(row["source"] for row in evidence_rows).value_counts().to_string())

Combined evidence rows: 7
presentation    6
transcript      1


## Run Experiment

Reuse the existing retrieval scorer and map each result back to its source citation.

In [16]:
def retrieve_one(query, rows):
    chunks = [
        TranscriptChunk(index, row["location"], row["location"], row["text"], 1)
        for index, row in enumerate(rows, start=1)
    ]
    result = search_chunks(query, chunks, top_k=1)[0]
    return rows[result.chunk.index - 1]

queries = [
    ("pptx evidence", "ย้ายงบประมาณไปสนับสนุนอะไร", pptx_rows),
    ("pdf evidence", "ทุกฝ่ายต้องทำอะไรก่อนส่งรายงาน", pdf_rows),
    ("audio evidence", "ความเสี่ยงเรื่องข้อมูลไม่ครบ", evidence_rows),
]
results = [
    {"case": case, "query": query, **retrieve_one(query, rows)}
    for case, query, rows in queries
]
results_df = pd.DataFrame(results)
print(results_df[["case", "query", "source", "filename", "location"]].to_string(index=False))

          case                          query       source                   filename      location
 pptx evidence     ย้ายงบประมาณไปสนับสนุนอะไร presentation meeting_budget_sample.pptx      slide-02
  pdf evidence ทุกฝ่ายต้องทำอะไรก่อนส่งรายงาน presentation  meeting_budget_sample.pdf       page-03
audio evidence   ความเสี่ยงเรื่องข้อมูลไม่ครบ   transcript           meeting_test.wav transcript-05


## Inspect Results

In [17]:
checks = [
    ("loads real PPTX 3 slides", len(pptx_rows) == 3),
    ("loads real PDF 3 pages", len(pdf_rows) == 3),
    ("retrieves PPTX slide 2 cite", results[0]["location"] == "slide-02"),
    ("retrieves PDF page 3 cite", results[1]["location"] == "page-03"),
    ("retrieves transcript-only cite", results[2]["location"] == "transcript-05"),
    ("PPTX result mentions training", "งานอบรมทีมงาน" in results[0]["text"]),
    ("PDF result mentions data checking", "ตรวจสอบข้อมูล" in results[1]["text"]),
    ("keeps sources distinguishable", {row["source"] for row in results} == {"presentation", "transcript"}),
]
checks_df = pd.DataFrame(checks, columns=["check", "passed"])
print(checks_df.to_string(index=False))
assert checks_df["passed"].all(), "Some presentation evidence checks failed"
print(f"Passed {checks_df['passed'].sum()}/{len(checks_df)} presentation evidence checks")

                            check  passed
         loads real PPTX 3 slides    True
           loads real PDF 3 pages    True
      retrieves PPTX slide 2 cite    True
        retrieves PDF page 3 cite    True
   retrieves transcript-only cite    True
    PPTX result mentions training    True
PDF result mentions data checking    True
    keeps sources distinguishable    True
Passed 8/8 presentation evidence checks


## What We Learned

- The real three-slide PPTX can be cited by slide and its PDF export can be cited by page.
- Thai questions retrieve the budget decision, assigned follow-up, and transcript-only risk evidence from the correct source.
- Canva needs no dedicated integration when users export PDF or PPTX.

## Common Problems

- PDF loading requires the native `pdftotext` command from Poppler.
- Image-only PDFs contain no extractable text; add OCR only after real files require it.
- The minimal PPTX parser follows numeric slide XML order; heavily reordered decks may need a full PPTX library.

In [18]:
print(f"PPTX tested: {pptx_path}")
print(f"PDF tested: {pdf_path}")

PPTX tested: /home/lataeq/Code/Saranote/outputs/manual_test/presentations/meeting_budget_sample.pptx
PDF tested: /home/lataeq/Code/Saranote/outputs/manual_test/presentations/meeting_budget_sample.pdf
